In [139]:
# Imports
import os
import datetime
import IPython
import IPython.display
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

In [140]:
# Mounting Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [141]:
%cd /content/drive/My Drive/TX_DATA

/content/drive/My Drive/TX_DATA


In [142]:
# Read in the merged station datasets
dfs = {}
for index in range(0, 6):
  df = pd.read_csv('merged_' + str(index + 1) + '.dat', sep=",", parse_dates=["Date"], index_col="Date")
  dfs['Station' + str(index + 1)] = df

In [143]:
for key in dfs.keys():
  print(len(dfs[key]))

58441
56366
58441
58441
58441
58441


In [144]:
for key in dfs.keys():
  print(dfs[key].isnull().sum())

Ppt_x            582
SWC_5            646
SWC_10           595
SWC_20           792
SWC_50           607
T_5              582
T_10             582
T_20             582
T_50             582
Flag               0
Ppt_y            583
Tair             588
RH               583
Windspeed        583
Winddirection    583
Srad             583
dtype: int64
Ppt_x            0
SWC_5            2
SWC_10           1
SWC_20           1
SWC_50           0
T_5              0
T_10             0
T_20             0
T_50             0
Flag             0
Ppt_y            0
Tair             0
RH               0
Windspeed        0
Winddirection    0
Srad             0
dtype: int64
Ppt_x             0
SWC_5             0
SWC_10            0
SWC_20            0
SWC_50            0
T_5               0
T_10              0
T_20              0
T_50              0
Flag              0
Ppt_y             0
Tair              0
RH                0
Windspeed         0
Winddirection     0
Srad             76
dtype: int64
P

#Functions to Create Linear Model for Missing Data

In [145]:
def target_data_split(frame, targ_field, feat_fields):
  df = frame.copy()
  df = df.dropna(subset=[targ_field])
  df = df.dropna(subset=feat_fields)
  target = df[targ_field].values
  x_data = df[feat_fields].values
  return (target, x_data)

In [146]:
def lin_model_missing_field(dfs, targ_field, feat_fields, targ_frame_key):
  LinModel = LinearRegression(fit_intercept=True)
  #fit model
  for key in dfs.keys():
    if key != targ_frame_key:
      target, x_data = target_data_split(dfs[key],targ_field, feat_fields)
      LinModel.fit(x_data, target)

  #predict
  x_data = dfs[targ_frame_key][feat_fields].dropna()
  y_pred = LinModel.predict(x_data.values)
  return pd.Series(data = y_pred, index = x_data.index)

In [147]:
def lin_model_missing_field(dfs, targ_field, feat_fields, targ_frame_key):
  LinModel = LinearRegression(fit_intercept=True)
  #fit model
  target_list = []
  x_data_list = []
  for key in dfs.keys():
    if key != targ_frame_key:
      target, x_data = target_data_split(dfs[key],targ_field, feat_fields)
      target_list.append(target)
      x_data_list.append(x_data)

  target = np.concatenate(target_list)
  x_data = np.concatenate(x_data_list)
  LinModel.fit(x_data, target)
  #predict
  x_data = dfs[targ_frame_key][feat_fields].dropna()
  y_pred = LinModel.predict(x_data.values)
  return pd.Series(data = y_pred, index = x_data.index)

# Handling Missing Values for SWC_50 and T_50 in Station 4


In [148]:
df_4 = dfs["Station4"]
df4_swc50 = df_4.pop("SWC_50")
df4_t50 = df_4.pop("T_50")

In [149]:
df_4.fillna(method='ffill', inplace=True)

In [150]:
#Station 4 SWC 50 Linear Prediction
targ_field = 'SWC_50'
feat_fields = ['SWC_5','SWC_10','SWC_20']
targ_frame_key = 'Station4'

swc_50_pred = lin_model_missing_field(dfs, targ_field, feat_fields, targ_frame_key)

print(swc_50_pred)

Date
2015-01-01 00:00:00    0.124141
2015-01-01 01:00:00    0.124141
2015-01-01 02:00:00    0.123385
2015-01-01 03:00:00    0.122011
2015-01-01 04:00:00    0.122011
                         ...   
2021-08-31 20:00:00    0.140630
2021-08-31 21:00:00    0.142340
2021-08-31 22:00:00    0.144470
2021-08-31 23:00:00    0.143714
2021-09-01 00:00:00    0.143714
Length: 58441, dtype: float64


In [151]:
#Station 4 T_50 Linear Predictions
targ_field = 'T_50'
feat_fields = ['T_5','T_10','T_20']
targ_frame_key = 'Station4'

t_50_pred = lin_model_missing_field(dfs, targ_field, feat_fields, targ_frame_key)

print(t_50_pred)

Date
2015-01-01 00:00:00     9.250857
2015-01-01 01:00:00     9.177386
2015-01-01 02:00:00     9.115200
2015-01-01 03:00:00     9.038530
2015-01-01 04:00:00     8.988516
                         ...    
2021-08-31 20:00:00    30.968002
2021-08-31 21:00:00    31.097018
2021-08-31 22:00:00    31.005883
2021-08-31 23:00:00    30.793907
2021-09-01 00:00:00    30.506262
Length: 58441, dtype: float64


In [152]:
#Add Predictions Back to df_4
df_4["SWC_50"] = swc_50_pred
df_4["T_50"] = t_50_pred
print(df_4)

                     Ppt_x  SWC_5  SWC_10  SWC_20    T_5   T_10   T_20  Flag  \
Date                                                                           
2015-01-01 00:00:00    0.0  0.232   0.253   0.199   2.60   4.05   8.15    51   
2015-01-01 01:00:00    0.0  0.232   0.253   0.199   2.60   4.01   8.08    51   
2015-01-01 02:00:00    0.0  0.231   0.253   0.199   2.59   3.97   8.03    51   
2015-01-01 03:00:00    0.0  0.232   0.254   0.199   2.61   3.94   7.95    51   
2015-01-01 04:00:00    0.0  0.232   0.254   0.199   2.64   3.93   7.90    51   
...                    ...    ...     ...     ...    ...    ...    ...   ...   
2021-08-31 20:00:00    0.0  0.113   0.137   0.142  34.52  34.02  30.25     3   
2021-08-31 21:00:00    0.0  0.112   0.137   0.143  33.28  33.38  30.43     3   
2021-08-31 22:00:00    0.0  0.112   0.136   0.143  32.26  32.74  30.52     3   
2021-08-31 23:00:00    0.0  0.111   0.136   0.143  31.45  32.15  30.58     3   
2021-09-01 00:00:00    0.0  0.111   0.13

In [153]:
df_4.fillna(method='ffill', inplace=True)

In [154]:
df_4.isnull().sum()

Ppt_x            0
SWC_5            0
SWC_10           0
SWC_20           0
T_5              0
T_10             0
T_20             0
Flag             0
Ppt_y            0
Tair             0
RH               0
Windspeed        0
Winddirection    0
Srad             0
SWC_50           0
T_50             0
dtype: int64

# Handling Missing Values for SWC_20 in Station 5

In [155]:
df_5 = dfs["Station5"]
swc_20 = df_5.pop("SWC_20")
df_5.fillna(method='ffill', inplace=True)
df_5

,Ppt_x,SWC_5,SWC_10,SWC_50,T_5,T_10,T_20,T_50,Flag,Ppt_y,Tair,RH,Windspeed,Winddirection,Srad
Date,,,,,,,,,,,,,,,
2015-01-01 00:00:00,0.0,0.232,0.266,0.266,4.35,5.24,0.00,12.40,176,0.0,-0.632,84.50,1.102,359.60,0.00
2015-01-01 01:00:00,0.0,0.232,0.266,0.266,4.25,5.16,0.00,12.37,176,0.0,-0.542,84.60,0.874,14.88,0.15
2015-01-01 02:00:00,0.0,0.232,0.266,0.266,4.21,5.09,0.00,12.35,176,0.0,-0.439,83.40,1.178,22.20,0.27
2015-01-01 03:00:00,0.0,0.232,0.265,0.266,4.19,5.06,0.00,12.31,176,0.0,-0.328,83.00,0.785,28.02,0.26
2015-01-01 04:00:00,0.0,0.232,0.265,0.266,4.18,5.00,0.00,12.31,176,0.0,-0.311,91.00,0.811,5.05,0.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08-31 20:00:00,0.0,0.175,0.171,0.185,33.26,32.34,31.01,28.62,0,0.0,29.810,59.85,1.728,152.70,0.01
2021-08-31 21:00:00,0.0,0.174,0.171,0.185,32.62,32.13,31.14,28.59,0,0.0,28.510,64.15,2.122,147.90,0.00
2021-08-31 22:00:00,0.0,0.174,0.170,0.185,32.03,31.85,31.18,28.59,0,0.0,27.770,68.13,2.449,154.50,0.00


In [156]:
null_index = swc_20.isnull().index

In [157]:
targ_field = 'SWC_20'
feat_fields = ['SWC_5','SWC_10','SWC_50']
targ_frame_key = 'Station5'
swc_20_pred = lin_model_missing_field(dfs, targ_field, feat_fields, targ_frame_key)


In [158]:
print(swc_20_pred)

Date
2015-01-01 00:00:00    0.267981
2015-01-01 01:00:00    0.267981
2015-01-01 02:00:00    0.267981
2015-01-01 03:00:00    0.267063
2015-01-01 04:00:00    0.267063
                         ...   
2021-08-31 20:00:00    0.172456
2021-08-31 21:00:00    0.172718
2021-08-31 22:00:00    0.171800
2021-08-31 23:00:00    0.172062
2021-09-01 00:00:00    0.171405
Length: 58441, dtype: float64


In [159]:
swc_20.loc[null_index] = swc_20_pred

<ipython-input-159-976e984d8f81>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  swc_20.loc[null_index] = swc_20_pred


In [160]:
df_5["SWC_20"] = swc_20
df_5.isnull().sum()

Ppt_x            0
SWC_5            0
SWC_10           0
SWC_50           0
T_5              0
T_10             0
T_20             0
T_50             0
Flag             0
Ppt_y            0
Tair             0
RH               0
Windspeed        0
Winddirection    0
Srad             0
SWC_20           0
dtype: int64

# Use Forward Fill to Remove Null Values in Other Station DFs

In [161]:
for key in dfs.keys():
  dfs[key].fillna(method='ffill', inplace=True)

In [162]:
for key in dfs.keys():
  print(len(dfs[key]))


58441
56366
58441
58441
58441
58441


In [163]:

for key in dfs.keys():
  print(dfs[key].isnull().sum())

Ppt_x            0
SWC_5            0
SWC_10           0
SWC_20           0
SWC_50           0
T_5              0
T_10             0
T_20             0
T_50             0
Flag             0
Ppt_y            0
Tair             0
RH               0
Windspeed        0
Winddirection    0
Srad             0
dtype: int64
Ppt_x            0
SWC_5            0
SWC_10           0
SWC_20           0
SWC_50           0
T_5              0
T_10             0
T_20             0
T_50             0
Flag             0
Ppt_y            0
Tair             0
RH               0
Windspeed        0
Winddirection    0
Srad             0
dtype: int64
Ppt_x            0
SWC_5            0
SWC_10           0
SWC_20           0
SWC_50           0
T_5              0
T_10             0
T_20             0
T_50             0
Flag             0
Ppt_y            0
Tair             0
RH               0
Windspeed        0
Winddirection    0
Srad             0
dtype: int64
Ppt_x            0
SWC_5            0
SWC_10     

#Remove Indexes Not Shared by All Stations

In [164]:
  index_union = pd.Index([])
  for station, df in dfs.items():
      index_union = index_union.union(df.index)

  index_int = index_union
  for station, df in dfs.items():
      index_int = index_int.intersection(df.index)
  for key in dfs.keys():
      dfs[key] = dfs[key].loc[index_int]

In [165]:
for key in dfs.keys():
  print(len(dfs[key]))

56366
56366
56366
56366
56366
56366


In [166]:
y1 = '2016-01-01 00:00:00'
y2 = '2017-01-01 00:00:00'
y3 = '2018-01-01 00:00:00'
y4 = '2019-01-01 00:00:00'
y5 = '2020-01-01 00:00:00'
y6 = '2021-01-01 00:00:00'
y7 = '2021-09-01 00:00:00'

In [167]:
for key in dfs.keys():
  print(key)
  print(len(dfs[key].loc[:y1]))
  print(len(dfs[key].loc[y1:y2]))
  print(len(dfs[key].loc[y2:y3]))
  print(len(dfs[key].loc[y3:y4]))
  print(len(dfs[key].loc[y4:y5]))
  print(len(dfs[key].loc[y5:y6]))
  print(len(dfs[key].loc[y6:y7]))
  print('\n')

Station1
8761
8785
8761
8761
8761
8785
3758


Station2
8761
8785
8761
8761
8761
8785
3758


Station3
8761
8785
8761
8761
8761
8785
3758


Station4
8761
8785
8761
8761
8761
8785
3758


Station5
8761
8785
8761
8761
8761
8785
3758


Station6
8761
8785
8761
8761
8761
8785
3758




In [170]:
#Notice 2016 and 2020 have an additional day from leep years so remove that as well

l1 = '2016-02-29 00:00:00'
l2 = '2016-02-29 23:00:00'

l3 = '2020-02-29 00:00:00'
l4 = '2020-02-29 23:00:00'

for key in dfs.keys():
  dfs[key].drop(dfs[key].loc[l1:l2].index, inplace=True)
  dfs[key].drop(dfs[key].loc[l3:l4].index, inplace=True)

In [171]:
for key in dfs.keys():
  print(key)
  print(len(dfs[key].loc[:y1]))
  print(len(dfs[key].loc[y1:y2]))
  print(len(dfs[key].loc[y2:y3]))
  print(len(dfs[key].loc[y3:y4]))
  print(len(dfs[key].loc[y4:y5]))
  print(len(dfs[key].loc[y5:y6]))
  print(len(dfs[key].loc[y6:y7]))
  print('\n')

Station1
8761
8761
8761
8761
8761
8761
3758


Station2
8761
8761
8761
8761
8761
8761
3758


Station3
8761
8761
8761
8761
8761
8761
3758


Station4
8761
8761
8761
8761
8761
8761
3758


Station5
8761
8761
8761
8761
8761
8761
3758


Station6
8761
8761
8761
8761
8761
8761
3758




In [172]:
# Now all the Years have the same number of days and we can save each df

for key in dfs.keys():
  dfs[key].to_csv(f'{key}_Final_Version.csv')